In [0]:
from pyspark.sql import functions as F
from pyspark.sql.types import (
    StructType,
    StructField,
    StringType,
    IntegerType,
    DoubleType
)



In [0]:
SOURCE_PATH = "/Volumes/transit_occupancy/reference/source_files"
TARGET_SCHEMA = "transit_occupancy.reference"

In [0]:
#We will not use inferSchema because explicit schemas are safer and more professional.

vehicle_schema = StructType([
    StructField("vehicle_id", StringType(), False),
    StructField("vehicle_type", StringType(), False),
    StructField("vehicle_capacity", IntegerType(), False),
    StructField("registration_number", StringType(), False),
    StructField("service_status", StringType(), False)
])

route_schema = StructType([
    StructField("route_id", StringType(), False),
    StructField("route_name", StringType(), False),
    StructField("source", StringType(), False),
    StructField("destination", StringType(), False),
    StructField("route_distance_km", DoubleType(), False)
])

stop_schema = StructType([
    StructField("stop_id", StringType(), False),
    StructField("stop_name", StringType(), False),
    StructField("latitude", DoubleType(), False),
    StructField("longitude", DoubleType(), False),
    StructField("zone", StringType(), False)
])

route_stop_schema = StructType([
    StructField("route_id", StringType(), False),
    StructField("stop_id", StringType(), False),
    StructField("stop_sequence", IntegerType(), False)
])

trip_schema = StructType([
    StructField("trip_id", StringType(), False),
    StructField("route_id", StringType(), False),
    StructField("vehicle_id", StringType(), False),
    StructField("scheduled_start_time", StringType(), False),
    StructField("scheduled_end_time", StringType(), False),
    StructField("direction", StringType(), False),
    StructField("trip_status", StringType(), False)
])

In [0]:
#Create a reusable loading function

def load_reference_csv(file_name, schema, table_name):
    file_path = f"{SOURCE_PATH}/{file_name}"
    target_table = f"{TARGET_SCHEMA}.{table_name}"

    dataframe = (
        spark.read
        .format("csv")
        .option("header", True)
        .schema(schema)
        .load(file_path)
        .withColumn("source_file", F.lit(file_name))
        .withColumn("loaded_at", F.current_timestamp())
    )

    (
        dataframe.write
        .format("delta")
        .mode("overwrite")
        .option("overwriteSchema", "true")
        .saveAsTable(target_table)
    )

    print(
        f"Loaded {dataframe.count()} rows "
        f"from {file_name} into {target_table}"
    )

    return dataframe

In [0]:
#laod the five files
vehicles_df = load_reference_csv(
    file_name="vehicles.csv",
    schema=vehicle_schema,
    table_name="vehicles"
)

routes_df = load_reference_csv(
    file_name="routes.csv",
    schema=route_schema,
    table_name="routes"
)

stops_df = load_reference_csv(
    file_name="stops.csv",
    schema=stop_schema,
    table_name="stops" 
)

route_stops_df = load_reference_csv(
    file_name="route_stops.csv",
    schema=route_stop_schema,
    table_name="route_stops"
)

trips_df = load_reference_csv(
    file_name="trips.csv",
    schema=trip_schema,
    table_name="trips"
)

In [0]:
#validate row counts
expected_counts = {
    "vehicles": 20,
    "routes": 5,
    "stops": 20,
    "route_stops": 26,
    "trips": 40
}

for table_name, expected_count in expected_counts.items():
    actual_count = spark.table(
        f"{TARGET_SCHEMA}.{table_name}"
    ).count()

    status = "PASS" if actual_count == expected_count else "FAIL"

    print(
        f"{table_name:<15} "
        f"expected={expected_count:<5} "
        f"actual={actual_count:<5} "
        f"status={status}"
    )

    assert actual_count == expected_count, (
        f"Row-count validation failed for {table_name}"
    )

In [0]:
#validate primary keys
primary_key_checks = {
    "vehicles": ["vehicle_id"],
    "routes": ["route_id"],
    "stops": ["stop_id"],
    "route_stops": ["route_id", "stop_id"],
    "trips": ["trip_id"]
}

for table_name, primary_keys in primary_key_checks.items():
    dataframe = spark.table(f"{TARGET_SCHEMA}.{table_name}")

    null_condition = None

    for column_name in primary_keys:
        condition = F.col(column_name).isNull()
        null_condition = (
            condition
            if null_condition is None
            else null_condition | condition
        )

    null_count = dataframe.filter(null_condition).count()

    duplicate_count = (
        dataframe
        .groupBy(*primary_keys)
        .count()
        .filter(F.col("count") > 1)
        .count()
    )

    status = (
        "PASS"
        if null_count == 0 and duplicate_count == 0
        else "FAIL"
    )

    print(
        f"{table_name:<15} "
        f"null_keys={null_count:<5} "
        f"duplicates={duplicate_count:<5} "
        f"status={status}"
    )

    assert null_count == 0
    assert duplicate_count == 0

In [0]:
#Validate foreign-key relationships
#Route-stop route validation

invalid_route_references = (
    route_stops_df.alias("rs")
    .join(
        routes_df.select("route_id").alias("r"),
        on="route_id",
        how="left_anti"
    )
)

print(
    "Invalid routes in route_stops:",
    invalid_route_references.count()
)

In [0]:
#Route-stop stop validation
invalid_stop_references = (
    route_stops_df.alias("rs")
    .join(
        stops_df.select("stop_id").alias("s"),
        on="stop_id",
        how="left_anti"
    )
)

print(
    "Invalid stops in route_stops:",
    invalid_stop_references.count()
)

In [0]:
#Trip route validation
invalid_trip_routes = (
    trips_df
    .join(
        routes_df.select("route_id"),
        on="route_id",
        how="left_anti"
    )
)

print(
    "Invalid route references in trips:",
    invalid_trip_routes.count()
)

In [0]:
#Trip vehicle validation
invalid_trip_vehicles = (
    trips_df
    .join(
        vehicles_df.select("vehicle_id"),
        on="vehicle_id",
        how="left_anti"
    )
)

print(
    "Invalid vehicle references in trips:",
    invalid_trip_vehicles.count()
)

In [0]:
#final assertions:
assert invalid_route_references.count() == 0
assert invalid_stop_references.count() == 0
assert invalid_trip_routes.count() == 0
assert invalid_trip_vehicles.count() == 0

print("All reference-data validations passed.")

In [0]:
%sql
show tables in transit_occupancy.reference;

In [0]:
%sql

SELECT *
FROM transit_occupancy.reference.trips
ORDER BY route_id, scheduled_start_time;